# 两阶段联合训练：去噪 → 反卷积

本 notebook 实现分阶段的联合训练策略：
- **第一阶段**：去噪权重大 (λ_denoise >> λ_deconv)，优先学习去噪
- **第二阶段**：反卷积权重大 (λ_deconv >> λ_denoise)，专注反卷积
- **优势**：比固定权重更稳定，比两阶段训练更端到端

训练流程:
```
阶段 1 (1-15 epochs): λ_denoise=10, λ_deconv=1  → 主攻去噪
阶段 2 (16-30 epochs): λ_denoise=1, λ_deconv=10  → 主攻反卷积
```

## 1. 导入依赖

In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用：{torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. 工具函数

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def percentile_norm(x, pmin=0.1, pmax=99.9, eps=1e-8):
    lo = np.percentile(x, pmin)
    hi = np.percentile(x, pmax)
    return np.clip((x - lo) / max(hi - lo, eps), 0.0, 1.0)

def load_image(path):
    arr = np.array(Image.open(path)).astype(np.float32)
    return arr.mean(axis=2) if arr.ndim == 3 else arr

def save_image(path, x):
    x = np.clip(x / max(x.max(), 1e-8), 0.0, 1.0)
    Image.fromarray((x * 65535.0).astype(np.uint16)).save(path)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def reflect_pad(x, multiple=16, margin=16):
    h, w = x.shape
    new_h = int(math.ceil(h / multiple) * multiple)
    new_w = int(math.ceil(w / multiple) * multiple)
    pad_h, pad_w = max(new_h - h, 0), max(new_w - w, 0)
    top, bottom = margin + pad_h // 2, margin + pad_h - pad_h // 2
    left, right = margin + pad_w // 2, margin + pad_w - pad_w // 2
    return np.pad(x, ((top, bottom), (left, right)), mode="reflect"), (top, bottom, left, right)

def crop_pad(x, padinfo, scale=1):
    top, bottom, left, right = [p * scale for p in padinfo]
    h, w = x.shape
    return x[top:h - bottom, left:w - right]

## 3. PSF 加载

In [ ]:
def gaussian_psf(size=25, sigma=2.0):
    if size % 2 == 0:
        size += 1
    r = size // 2
    y, x = np.mgrid[-r:r + 1, -r:r + 1]
    psf = np.exp(-(x ** 2 / (2 * sigma ** 2) + y ** 2 / (2 * sigma ** 2)))
    psf = psf.astype(np.float32)
    psf /= np.maximum(psf.sum(), 1e-8)
    return psf

def load_psf_from_file(psf_path, size=25):
    """从文件加载真实 PSF，裁剪中心区域"""
    psf = load_image(psf_path)
    
    h, w = psf.shape
    center_y, center_x = h // 2, w // 2
    half = size // 2
    
    top = max(0, center_y - half)
    bottom = min(h, center_y + half + 1)
    left = max(0, center_x - half)
    right = min(w, center_x + half + 1)
    
    psf = psf[top:bottom, left:right]
    
    if psf.shape[0] < size or psf.shape[1] < size:
        pad_h = size - psf.shape[0]
        pad_w = size - psf.shape[1]
        psf = np.pad(psf, ((0, pad_h), (0, pad_w)), mode='constant')
    
    psf /= np.maximum(psf.sum(), 1e-8)
    return psf

def make_psf_tensor(mode, img, size=25, sigma=None, device=None, psf_path=None):
    if mode == "load":
        psf_path = psf_path or "datasets/Microtubule/PSF/psf_emLambda525_dxy0.0313_NA1.3.tif"
        psf = load_psf_from_file(psf_path, size)
    elif mode == "estimate":
        sigma = sigma or 2.0
        psf = gaussian_psf(size, sigma)
    else:
        sigma = sigma or 2.0
        psf = gaussian_psf(size, sigma)
    
    psf_t = torch.from_numpy(psf)[None, None].to(device)
    return psf_t / psf_t.sum().clamp_min(1e-8)

## 4. 数据加载

In [ ]:
class PseudoPairDataset(Dataset):
    """从单张图像生成伪配对数据（添加噪声）"""
    def __init__(self, img, patch_size=128, n_samples=2000):
        self.img = img
        self.patch_size = patch_size
        self.n_samples = n_samples
        self.h, self.w = img.shape
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        h_start = np.random.randint(0, self.h - self.patch_size)
        w_start = np.random.randint(0, self.w - self.patch_size)
        
        patch = self.img[h_start:h_start + self.patch_size, w_start:w_start + self.patch_size]
        
        noisy = patch + np.random.normal(0, 0.1, patch.shape).astype(np.float32)
        noisy = np.clip(noisy, 0, 1)
        
        return (
            torch.from_numpy(noisy[None]),
            torch.from_numpy(patch[None])
        )

img = percentile_norm(load_image("datasets/Microtubule/train_data/01.tif"))
dataset = PseudoPairDataset(img, patch_size=128, n_samples=10)
inp, gt = dataset[0]
print(f"输入形状：{inp.shape}, 目标形状：{gt.shape}")

## 5. 联合模型定义

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_conv=3):
        super().__init__()
        layers = []
        for i in range(n_conv):
            layers.append(nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1))
            layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class Encoder(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        ch = in_ch
        for i in range(depth):
            out_ch = base_ch * (2 ** i)
            self.blocks.append(ConvBlock(ch, out_ch, n_conv))
            self.pools.append(nn.MaxPool2d(2))
            ch = out_ch
        self.out_ch = ch

    def forward(self, x):
        skips = []
        for blk, pool in zip(self.blocks, self.pools):
            x = blk(x)
            skips.append(x)
            x = pool(x)
        return x, skips


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_conv=3):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        layers = [nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True)]
        cur = out_ch
        for _ in range(n_conv - 1):
            next_ch = max(out_ch // 2, 1)
            layers.extend([nn.Conv2d(cur, next_ch, 3, padding=1), nn.ReLU(inplace=True)])
            cur = next_ch
        self.conv = nn.Sequential(*layers)
        self.out_ch = cur

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.conv(torch.cat([x, skip], dim=1))


class UNetStage(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.encoder = Encoder(in_ch, base_ch, depth, n_conv)
        mid_ch = self.encoder.out_ch * 2
        self.mid = nn.Sequential(
            nn.Conv2d(self.encoder.out_ch, mid_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, self.encoder.out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.decoders = nn.ModuleList()
        cur_ch = self.encoder.out_ch
        for i in reversed(range(depth)):
            skip_ch = base_ch * (2 ** i)
            block = DecoderBlock(cur_ch, skip_ch, skip_ch, n_conv)
            self.decoders.append(block)
            cur_ch = block.out_ch
        self.out_ch = cur_ch

    def forward(self, x):
        x, skips = self.encoder(x)
        x = self.mid(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return x


class JointDenoiseDeconvNet(nn.Module):
    """
    联合去噪 + 反卷积网络
    
    架构:
        输入 (噪声) → DenoiseUNet → 去噪图像 → DeconvUNet → 反卷积结果
    """
    def __init__(self, base_ch=32, depth=4, n_conv=3, upsample=True):
        super().__init__()
        self.upsample = upsample
        
        # 去噪阶段
        self.denoise = UNetStage(1, base_ch, depth, n_conv)
        self.out_denoise = nn.Conv2d(self.denoise.out_ch, 1, 3, padding=1)
        
        # 反卷积阶段
        self.deconv = UNetStage(1, base_ch, depth, n_conv)
        self.refine1 = nn.Conv2d(self.deconv.out_ch, 128, 3, padding=1)
        self.refine2 = nn.Conv2d(128, 128, 3, padding=1)
        self.out_deconv = nn.Conv2d(128, 1, 3, padding=1)
    
    def forward(self, x):
        # 去噪阶段
        feat_denoise = self.denoise(x)
        denoised = F.relu(self.out_denoise(feat_denoise))
        
        # 反卷积阶段 (以去噪结果为输入)
        feat_deconv = self.deconv(denoised)
        if self.upsample:
            feat_deconv = F.interpolate(feat_deconv, scale_factor=2, mode="nearest")
        deconv = F.relu(self.out_deconv(F.relu(self.refine2(F.relu(self.refine1(feat_deconv))))))
        
        return denoised, deconv

## 6. 损失函数

In [ ]:
class HessianLoss(nn.Module):
    """Hessian 正则，抑制伪影"""
    def forward(self, x):
        dxx = x[:, :, :, :-2] - 2 * x[:, :, :, 1:-1] + x[:, :, :, 2:]
        dyy = x[:, :, :-2, :] - 2 * x[:, :, 1:-1, :] + x[:, :, 2:, :]
        dxy = x[:, :, 1:, 1:] - x[:, :, :-1, 1:] - x[:, :, 1:, :-1] + x[:, :, :-1, :-1]
        return (dxx.pow(2).mean() + dyy.pow(2).mean() + dxy.pow(2).mean()) / 3.0


class TVLoss(nn.Module):
    """Total Variation 正则"""
    def forward(self, x):
        dx = x[:, :, :, 1:] - x[:, :, :, :-1]
        dy = x[:, :, 1:, :] - x[:, :, :-1, :]
        return dx.pow(2).mean() + dy.pow(2).mean()


class DeconvLoss(nn.Module):
    """反卷积损失：PSF 卷积重建 + 正则"""
    def __init__(self, psf, upsample=True, hess_w=0.02, tv_w=0.0, l1_w=0.0):
        super().__init__()
        self.register_buffer("psf", psf)
        self.upsample = upsample
        self.hess_w, self.tv_w, self.l1_w = hess_w, tv_w, l1_w
        self.hess = HessianLoss()
        self.tv = TVLoss()

    def forward(self, y_true, y_pred):
        k = self.psf.shape[-1]
        pad = k // 2
        y_conv = F.conv2d(F.pad(y_pred, (pad, pad, pad, pad), mode="reflect"), self.psf, padding=0)
        if self.upsample:
            y_conv = F.interpolate(y_conv, size=y_true.shape[-2:], mode="bilinear", align_corners=False)
        
        loss = F.l1_loss(y_conv, y_true)
        if self.hess_w > 0:
            loss = loss + self.hess_w * self.hess(y_pred)
        if self.tv_w > 0:
            loss = loss + self.tv_w * self.tv(y_pred)
        if self.l1_w > 0:
            loss = loss + self.l1_w * y_pred.abs().mean()
        return loss


class JointLoss(nn.Module):
    """
    联合损失：去噪损失 + 反卷积损失
    
    total_loss = λ_denoise * L_denoise + λ_deconv * L_deconv
    """
    def __init__(self, psf, upsample=True, 
                 lambda_denoise=1.0, lambda_deconv=1.0,
                 hess_w=0.02, tv_w=0.0):
        super().__init__()
        self.lambda_denoise = lambda_denoise
        self.lambda_deconv = lambda_deconv
        
        self.denoise_criterion = nn.L1Loss()
        self.deconv_criterion = DeconvLoss(psf, upsample, hess_w, tv_w)
    
    def forward(self, y_true, denoised, deconv):
        loss_denoise = self.denoise_criterion(denoised, y_true)
        loss_deconv = self.deconv_criterion(y_true, deconv)
        total_loss = self.lambda_denoise * loss_denoise + self.lambda_deconv * loss_deconv
        return total_loss, loss_denoise, loss_deconv

## 7. 两阶段训练函数

In [ ]:
def train_joint_two_stage(
    image_path="datasets/Microtubule/train_data/01.tif",
    psf_path="datasets/Microtubule/PSF/psf_emLambda525_dxy0.0313_NA1.3.tif",
    out_dir="./output_joint_two_stage",
    psf_size=25,
    patch_size=128,
    n_samples=2000,
    batch_size=4,
    epochs_stage1=15,
    epochs_stage2=15,
    lr=1e-4,
    lr_step=10,
    lr_gamma=0.5,
    depth=4,
    n_conv=3,
    base_ch=32,
    upsample=True,
    # 阶段 1: 去噪为主
    lambda_denoise_s1=10.0,
    lambda_deconv_s1=1.0,
    # 阶段 2: 反卷积为主
    lambda_denoise_s2=1.0,
    lambda_deconv_s2=10.0,
    hess_w=0.02,
    tv_w=0.0,
    seed=42,
):
    """
    两阶段联合训练:
    阶段 1 (1-15 epochs): λ_denoise=10, λ_deconv=1  → 主攻去噪
    阶段 2 (16-30 epochs): λ_denoise=1, λ_deconv=10  → 主攻反卷积
    """
    set_seed(seed)
    ensure_dir(out_dir)
    
    print(f"设备：{device}")
    print(f"加载图像：{image_path}")
    img = percentile_norm(load_image(image_path))
    
    print(f"加载 PSF: {psf_path} ({psf_size}x{psf_size})")
    psf = make_psf_tensor("load", img, psf_size, device=device, psf_path=psf_path)
    
    print("创建数据集...")
    dataset = PseudoPairDataset(img, patch_size, n_samples)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    print("初始化联合模型...")
    model = JointDenoiseDeconvNet(base_ch, depth, n_conv, upsample).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=lr_step, gamma=lr_gamma)
    
    best_path = os.path.join(out_dir, "best_joint_two_stage.pt")
    best_loss = float("inf")
    history = {'total': [], 'denoise': [], 'deconv': [], 'phase': []}
    
    total_epochs = epochs_stage1 + epochs_stage2
    
    print("=" * 70)
    print(f"开始两阶段联合训练，共 {total_epochs} epochs")
    print(f"阶段 1 (1-{epochs_stage1}): λ_denoise={lambda_denoise_s1}, λ_deconv={lambda_deconv_s1}")
    print(f"阶段 2 ({epochs_stage1+1}-{total_epochs}): λ_denoise={lambda_denoise_s2}, λ_deconv={lambda_deconv_s2}")
    print("=" * 70)
    
    model.train()
    for epoch in range(total_epochs):
        # 根据阶段调整损失权重
        if epoch < epochs_stage1:
            # 阶段 1: 去噪为主
            lambda_denoise = lambda_denoise_s1
            lambda_deconv = lambda_deconv_s1
            phase = 1
        else:
            # 阶段 2: 反卷积为主
            lambda_denoise = lambda_denoise_s2
            lambda_deconv = lambda_deconv_s2
            phase = 2
        
        # 更新损失函数的权重
        loss_fn = JointLoss(psf, upsample, lambda_denoise, lambda_deconv, hess_w, tv_w)
        
        losses_total = []
        losses_denoise = []
        losses_deconv = []
        
        for inp, gt in loader:
            inp, gt = inp.to(device), gt.to(device)
            
            denoised, deconv = model(inp)
            total_loss, loss_denoise, loss_deconv = loss_fn(gt, denoised, deconv)
            
            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            optimizer.step()
            
            losses_total.append(total_loss.item())
            losses_denoise.append(loss_denoise.item())
            losses_deconv.append(loss_deconv.item())
        
        scheduler.step()
        
        mean_total = np.mean(losses_total)
        mean_denoise = np.mean(losses_denoise)
        mean_deconv = np.mean(losses_deconv)
        lr_now = optimizer.param_groups[0]["lr"]
        
        history['total'].append(mean_total)
        history['denoise'].append(mean_denoise)
        history['deconv'].append(mean_deconv)
        history['phase'].append(phase)
        
        # 打印训练进度
        marker = "★" if phase == 1 else "☆"
        print(f"{marker} 阶段{phase} | epoch {epoch+1:03d}/{total_epochs:03d} | "
              f"total={mean_total:.6f} | denoise={mean_denoise:.6f} | deconv={mean_deconv:.6f} | "
              f"lr={lr_now:.2e} | λ_d={lambda_denoise:.1f}, λ_c={lambda_deconv:.1f}")
        
        if mean_total < best_loss:
            best_loss = mean_total
            torch.save({
                "model": model.state_dict(),
                "psf": psf.cpu(),
                "cfg": {"base_ch": base_ch, "depth": depth, "n_conv": n_conv, "upsample": upsample,
                       "lambda_denoise_s1": lambda_denoise_s1, "lambda_deconv_s1": lambda_deconv_s1,
                       "lambda_denoise_s2": lambda_denoise_s2, "lambda_deconv_s2": lambda_deconv_s2,
                       "best_epoch": epoch+1, "best_phase": phase},
                "best_loss": best_loss
            }, best_path)
    
    psf_np = psf.squeeze().cpu().numpy()
    save_image(os.path.join(out_dir, "psf_used.tif"), psf_np / psf_np.max())
    
    # 绘制损失曲线
    plt.figure(figsize=(12, 5))
    
    # 总损失
    plt.subplot(1, 2, 1)
    plt.plot(history['total'], 'k-', linewidth=2, label='Total Loss')
    plt.axvline(x=epochs_stage1-0.5, color='gray', linestyle='--', linewidth=2, label='Phase Boundary')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Two-Stage Joint Training Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 分项损失
    plt.subplot(1, 2, 2)
    plt.plot(history['denoise'], 'g-', linewidth=2, label='Denoise Loss')
    plt.plot(history['deconv'], 'r-', linewidth=2, label='Deconv Loss')
    plt.axvline(x=epochs_stage1-0.5, color='gray', linestyle='--', linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Component Losses")
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "loss_curve_two_stage.png"), dpi=150)
    plt.show()
    
    print(f"\n训练完成！最佳模型在第 {history['phase'].index(best_loss)} 阶段保存")
    print(f"模型已保存：{best_path}")
    return best_path, history

## 8. 推理函数

In [ ]:
@torch.no_grad()
def infer_joint(image_path, checkpoint_path, out_dir="./output_joint_two_stage"):
    ensure_dir(out_dir)
    
    print(f"加载模型：{checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = ckpt["cfg"]
    
    model = JointDenoiseDeconvNet(
        cfg["base_ch"], cfg["depth"], cfg["n_conv"], cfg["upsample"]
    ).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    
    print(f"处理图像：{image_path}")
    img = percentile_norm(load_image(image_path))
    img_pad, padinfo = reflect_pad(img, multiple=2 ** cfg["depth"], margin=16)
    x = torch.from_numpy(img_pad[None, None]).float().to(device)
    
    denoised, deconv = model(x)
    
    scale = 2 if cfg["upsample"] else 1
    denoised_np = crop_pad(denoised.squeeze().cpu().numpy(), padinfo, scale=1)
    deconv_np = crop_pad(deconv.squeeze().cpu().numpy(), padinfo, scale=scale)
    
    name = Path(image_path).stem
    denoised_path = os.path.join(out_dir, f"{name}_denoised_joint.tif")
    deconv_path = os.path.join(out_dir, f"{name}_deconved_joint.tif")
    
    save_image(denoised_path, percentile_norm(denoised_np))
    save_image(deconv_path, percentile_norm(deconv_np))
    
    print(f"去噪结果：min={denoised_np.min():.4f}, max={denoised_np.max():.4f}")
    print(f"反卷积结果：min={deconv_np.min():.4f}, max={deconv_np.max():.4f}")
    print(f"已保存：{denoised_path}, {deconv_path}")
    
    plt.figure(figsize=(18, 5))
    
    plt.subplot(1, 4, 1)
    plt.imshow(img, cmap="gray")
    plt.title("Input (Noisy)")
    plt.axis("off")
    
    plt.subplot(1, 4, 2)
    plt.imshow(img, cmap="gray")
    plt.title("Input (Zoom)")
    plt.axis("off")
    
    plt.subplot(1, 4, 3)
    plt.imshow(denoised_np, cmap="gray")
    plt.title("Denoised (Intermediate)")
    plt.axis("off")
    
    plt.subplot(1, 4, 4)
    plt.imshow(deconv_np, cmap="gray")
    plt.title("Deconvolved (Final)")
    plt.axis("off")
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{name}_joint_result.png"), dpi=150)
    plt.show()
    
    return denoised_path, deconv_path, denoised_np, deconv_np

## 9. 配置参数

In [ ]:
# 路径配置
IMAGE_PATH = "datasets/Microtubule/train_data/01.tif"
PSF_PATH = "datasets/Microtubule/PSF/psf_emLambda525_dxy0.0313_NA1.3.tif"
OUT_DIR = "./output_joint_two_stage"

# 数据集配置
PATCH_SIZE = 128
N_SAMPLES = 2000
BATCH_SIZE = 4

# 训练配置
EPOCHS_STAGE1 = 15   # 阶段 1: 去噪为主
EPOCHS_STAGE2 = 15   # 阶段 2: 反卷积为主
LR = 1e-4
LR_STEP = 10
LR_GAMMA = 0.5

# 模型配置
DEPTH = 4
N_CONV = 3
BASE_CH = 32
UPSAMPLE = True

# 损失权重 (关键参数！)
# 阶段 1: 主攻去噪
LAMBDA_DENOISE_S1 = 10.0   # 去噪权重大
LAMBDA_DECONV_S1 = 1.0     # 反卷积权重小

# 阶段 2: 主攻反卷积
LAMBDA_DENOISE_S2 = 1.0    # 去噪权重小
LAMBDA_DECONV_S2 = 10.0    # 反卷积权重大

# 正则化
HESS_W = 0.05
TV_W = 0.0

SEED = 42

print("两阶段联合训练配置已设置完成")
print(f"阶段 1 (1-{EPOCHS_STAGE1}): λ_denoise={LAMBDA_DENOISE_S1}, λ_deconv={LAMBDA_DECONV_S1}")
print(f"阶段 2 ({EPOCHS_STAGE1+1}-{EPOCHS_STAGE1+EPOCHS_STAGE2}): λ_denoise={LAMBDA_DENOISE_S2}, λ_deconv={LAMBDA_DECONV_S2}")

## 10. 开始训练

In [ ]:
# 两阶段联合训练
checkpoint_path, history = train_joint_two_stage(
    image_path=IMAGE_PATH,
    psf_path=PSF_PATH,
    out_dir=OUT_DIR,
    psf_size=25,
    patch_size=PATCH_SIZE,
    n_samples=N_SAMPLES,
    batch_size=BATCH_SIZE,
    epochs_stage1=EPOCHS_STAGE1,
    epochs_stage2=EPOCHS_STAGE2,
    lr=LR,
    lr_step=LR_STEP,
    lr_gamma=LR_GAMMA,
    depth=DEPTH,
    n_conv=N_CONV,
    base_ch=BASE_CH,
    upsample=UPSAMPLE,
    lambda_denoise_s1=LAMBDA_DENOISE_S1,
    lambda_deconv_s1=LAMBDA_DECONV_S1,
    lambda_denoise_s2=LAMBDA_DENOISE_S2,
    lambda_deconv_s2=LAMBDA_DECONV_S2,
    hess_w=HESS_W,
    tv_w=TV_W,
    seed=SEED,
)

## 11. 测试推理

In [ ]:
# 使用训练好的模型进行推理
denoised_path, deconv_path, denoised_np, deconv_np = infer_joint(
    image_path=IMAGE_PATH,
    checkpoint_path=checkpoint_path,
    out_dir=OUT_DIR,
)

## 12. 结果对比

In [ ]:
original = load_image(IMAGE_PATH)
original_norm = percentile_norm(original)
denoise_result_norm = percentile_norm(denoised_np)
deconv_result_norm = percentile_norm(deconv_np)

plt.figure(figsize=(18, 5))

plt.subplot(1, 4, 1)
plt.imshow(original_norm, cmap="gray")
plt.title(f"Original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(denoise_result_norm, cmap="gray")
plt.title(f"Denoised (Two-Stage)")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(deconv_result_norm, cmap="gray")
plt.title(f"Deconvolved (Two-Stage)")
plt.axis("off")

center_y = original.shape[0] // 2
x = np.arange(original.shape[1])
plt.subplot(1, 4, 4)
plt.plot(x, original_norm[center_y, :], 'b-', label='Original', alpha=0.7)
plt.plot(x, denoise_result_norm[center_y, :], 'g-', label='Denoised', alpha=0.7)

if deconv_np.shape[0] == original.shape[0] * 2:
    center_y_deconv = center_y * 2
    x_deconv = np.arange(deconv_np.shape[1])
    plt.plot(x_deconv, deconv_result_norm[center_y_deconv, :], 'r-', label='Deconvolved', alpha=0.7)
else:
    plt.plot(x, deconv_result_norm[center_y, :], 'r-', label='Deconvolved', alpha=0.7)

plt.xlabel("Pixel")
plt.ylabel("Normalized Intensity")
plt.title("Horizontal Profile")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "final_comparison_two_stage.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\n所有结果已保存到：{OUT_DIR}")